In [88]:
import torch
import torch.nn.functional as f
import torchvision
from torchvision import transforms
from torch.utils.data import dataloader
from torch import nn

In [89]:
class Unet_MRI(nn.Module):
    def __init__(self,para):
        super(Unet_MRI,self).__init__()
        self.para=para
        self.downsapling_channel=para['Downsampling_channel']
        self.upsampling_channel=para['Upsampling_channel']
        self.kernel_size=para['Kernel_size']
        self.activation_func=para['activation_func']

In [90]:
class convolution_block(nn.Module):
    def __init__(self, input_channel,output_channel):
        super().__init__()
        self.block=nn.Sequential(
            nn.Conv2d(in_channels=input_channel,out_channels=output_channel,kernel_size=3,padding=1,bias=False),
            nn.BatchNorm2d(output_channel),
            nn.Conv2d(in_channels=output_channel,out_channels=output_channel,kernel_size=3,padding=1,bias=False),
            nn.BatchNorm2d(output_channel),
            nn.ReLU(inplace=True)
        )
    def forward(self,x):
        x=self.block(x)
        return x
    
    

In [98]:
class Downsampling(nn.Module):
    def __init__(self,input_channels,output_channels):
        super().__init__()
        self.conv_block=convolution_block(input_channels,output_channels)
        self.max_pooling=nn.MaxPool2d((2,2),stride=2)
    def forward(self,x):
        x=self.conv_block(x)
        c=self.max_pooling(x)
        return x,c

In [92]:
class Upsampling(nn.Module):
    def __init__(self,input_channels,output_channels):
        super().__init__()
        self.up_conv_block=nn.ConvTranspose2d(in_channels=input_channels,out_channels=output_channels,kernel_size=2,stride=2,padding=0)
        self.conv_block=convolution_block(output_channels+output_channels,output_channels)
    def forward(self,x,concatination_feature):
        x=self.up_conv_block(x)
        x=torch.cat([x,concatination_feature],axis=1)
        x=self.conv_block(x)
        return x

In [100]:

class UNET_Architecture(nn.Module):
  def __init__(self):
    super().__init__()

    self.encoder_1 = Downsampling(1,64)
    self.encoder_2 = Downsampling(64,128)
    self.encoder_3 = Downsampling(128,256)
    self.encoder_4 = Downsampling(256,512)

    self.bottleneck = convolution_block(512,1024)

    self.decoder_1 = Upsampling(1024,512)
    self.decoder_2 = Upsampling(512,256)
    self.decoder_3 = Upsampling(256,128)
    self.decoder_4 = Upsampling(128,64)

    self.output_layer = nn.Conv2d(64,1,kernel_size=1)

  def forward(self,x):
    #contracting path
    x1,p1 = self.encoder_1(x)
    x2,p2 = self.encoder_2(p1)
    x3,p3 = self.encoder_3(p2)
    x4,p4 = self.encoder_4(p3)

    #bottleneck
    b = self.bottleneck(p4)

    #expanding path
    d1 = self.decoder_1(b,x4) #use of skip connection
    d2 = self.decoder_2(d1,x3) #use of skip connection
    d3 = self.decoder_3(d2,x2) #use of skip connection
    d4 = self.decoder_4(d3,x1) #use of skip connection

    #output layer
    output = self.output_layer(d4)

    return output

In [101]:

model = UNET_Architecture() #create an instance of the model
dummy_input = torch.randn(1, 1,512 , 512)
print(f"Input tensor shape: {dummy_input.shape}")
output = model(dummy_input)
print(f"Output tensor shape: {output.shape}")



Input tensor shape: torch.Size([1, 1, 512, 512])
Output tensor shape: torch.Size([1, 1, 512, 512])
